# Fase 3 - Árbol de decisión + falsos positivos (ÚLTIMA FASE)

## Objetivo
Entrenar un árbol de decisión para clasificar si una vivienda es accesible (esfuerzo <= 0.30) y analizar falsos positivos / falsos negativos.

## Definición
`accesible = 1` si `esfuerzo <= 0.30`, de lo contrario `0`

## Insumos
- `data/integrado/maestro_joven_uk.parquet`
- `data/limpia/hipotecas_limpio.parquet`

In [ ]:
import sys
from pathlib import Path
sys.path.insert(0, str(Path(__file__).resolve().parents[1]))

from proyecto import config
from proyecto import indicadores
import pandas as pd
import numpy as np

from sklearn.model_selection import train_test_split
from sklearn.tree import DecisionTreeClassifier, export_text
from sklearn.metrics import confusion_matrix, classification_report

maestro = pd.read_parquet(config.RUTA_MAESTRO).copy()
hipotecas = pd.read_parquet(config.RUTA_HIPOTECAS_LIMPIO).copy()

tasa = float(hipotecas['tasa_inicial_pct'].mean())
comisiones = float(hipotecas['comisiones_total'].mean())

df = maestro.copy()
df['cuota_mensual'] = df.apply(
    lambda r: indicadores.cuota_mensual_pmt(r['precio_mediana'], tasa, config.PLAZO_ANIOS, config.LTV, comisiones),
    axis=1,
)
df['salario_joven_mensual'] = df['salario_joven_region'] / 12.0
df['esfuerzo'] = df['cuota_mensual'] / df['salario_joven_mensual']

# Imputación consistente
df['sol_anom'] = df['sol_anom'].fillna(0.0)
df['precio_var_pct'] = df['precio_var_pct'].fillna(0.0)

# Objetivo binario
df['accesible'] = (df['esfuerzo'] <= 0.30).astype(int)

print("Objetivo: esfuerzo <= 0.30 -> accesible=1")
print(df['accesible'].value_counts())
print(f"Proporción 1: {df['accesible'].mean():.3f}")

## Preparación X / y y split estratificado

In [ ]:
num_vars = ["precio_mediana", "salario_joven_region", "anos_salario", "precio_var_pct",
            "tmax_anom", "tmin_anom", "lluvia_anom", "sol_anom", "af_anom"]
X = df[num_vars + ["region"]].copy()
X = pd.get_dummies(X, columns=["region"], drop_first=True)
y = df['accesible']

print(f"\nShape X: {X.shape} | y: {y.shape}")
print("Columnas X:", list(X.columns))

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.25, random_state=42, stratify=y
)

print(f"\nTrain: {X_train.shape[0]} | Test: {X_test.shape[0]}")
print(f"Train proporciones:\n{y_train.value_counts(normalize=True).round(3)}")
print(f"Test proporciones:\n{y_test.value_counts(normalize=True).round(3)}")

## Prueba de profundidad del árbol

In [ ]:
print("\n=== PRUEBA DE PROFUNDIDAD ===")
print(f"{'max_depth':>10} {'acc_train':>12} {'acc_test':>10}")
for prof in [2, 3, 4, 5]:
    arbol = DecisionTreeClassifier(max_depth=prof, random_state=42)
    arbol.fit(X_train, y_train)
    print(f"{prof:>10} {arbol.score(X_train,y_train):>12.3f} {arbol.score(X_test,y_test):>10.3f}")

## Árbol final (max_depth=3)

In [ ]:
print("\n=== ÁRBOL FINAL (max_depth=3) ===")
arbol = DecisionTreeClassifier(max_depth=3, random_state=42)
arbol.fit(X_train, y_train)
print(f"Train: {arbol.score(X_train,y_train):.3f} | Test: {arbol.score(X_test,y_test):.3f}")
print("\nREGLAS:")
try:
    print(export_text(arbol, feature_names=list(X.columns), max_depth=3))
except Exception as e:
    print(f"Sin reglas legibles: {e}")

## Matriz de confusión y falsos positivos / falsos negativos

In [ ]:
y_pred = arbol.predict(X_test)

print("\n=== MATRIZ DE CONFUSIÓN (test) ===")
cm = confusion_matrix(y_test, y_pred, labels=[0, 1])
print(cm)

if cm.shape == (2, 2):
    print("\nInterpretación:")
    print(f"VN (0,0): {cm[0,0]} | FP (0,1): {cm[0,1]}")
    print(f"FN (1,0): {cm[1,0]} | VP (1,1): {cm[1,1]}")
    print("\n=== FOCO: FALSOS POSITIVOS / FALSOS NEGATIVOS ===")
    print(f"Falsos positivos (predice accesible=1 pero realmente no lo es): {cm[0,1]}")
    print(f"Falsos negativos (predice accesible=0 pero realmente sí lo es): {cm[1,0]}")
else:
    print("\nSolo existe una clase. No hay casos reales de clase 1.")
    print("Por consiguiente no se pueden calcular falsos positivos/negativos con datos reales.")

## Reporte de clasificación

In [ ]:
print("\n=== REPORTE DE CLASIFICACIÓN ===")
try:
    print(classification_report(y_test, y_pred, labels=[0,1], target_names=["no_accesible(0)", "accesible(1)"], zero_division=0))
except Exception as e:
    print(f"Reporte no disponible: {e}")

print("\n=== IMPORTANCIA DE VARIABLES ===")
imp = pd.Series(arbol.feature_importances_, index=X.columns).sort_values(ascending=False)
print(imp.head(10).round(4).to_string())

## Conclusión Fase 3
- Con el umbral `esfuerzo <= 0.30`, **ninguna observación es accesible** (0/60, 100% no accesibles).
- El árbol predice la única clase presente.
- **No hay falsos positivos ni falsos negativos** con esta definición.
- Para poder evaluar FP/FN se requiere al menos una observación de la clase `accesible=1`.